# 02. Phân Tích Các Tác Tử Cổ Điển (Level 1 – Level 4)

**Mục tiêu:**
1. Kiểm chứng hàm lượng giá tĩnh (`evaluate_board`: Material + PST Michniewski + King Safety + Mobility + Cấu trúc Tốt + Xe cột mở + Cặp Tượng).
2. Phân tích hành vi ra quyết định và tốc độ suy luận (`NPS`/`SPS`, Latency) của **Level 1 (Random)**, **Level 2 (Heuristic 6 Tiers + SEE)**, **Level 3 (Alpha-Beta + PVS + Aspiration + TT + Quiescence/SEE)** và **Level 4 (Vanilla MCTS UCT)**.
3. **Thí nghiệm đối chứng Level 4:** So sánh trực tiếp **`rollout_mode="see_tactical"`** (rollout có luật SEE) vs **`rollout_mode="uniform"`** (rollout ngẫu nhiên hoàn toàn) về tỷ lệ thắng, độ trễ và tốc độ `SPS`.

In [1]:
from functools import partial
import json
from pathlib import Path
import sys

import chess
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from chess_ai.agents.level4_mcts import Level4MCTSAgent
from chess_ai.agents.registry import create_agent
from chess_ai.arena.tournament import run_paired_matchup
from chess_ai.core.chess_env import ChessEnv
from chess_ai.core.evaluation import (
    bishop_pair_score,
    evaluate_board,
    pawn_structure_score,
    rook_file_score,
)
from chess_ai.utils.epd import load_epd_file

## 1. Phân Tích Hàm Lượng Giá Tĩnh & Các Thành Phần Vị Trí

In [2]:
sample_fens = {
    "Khai cuộc chuẩn": chess.STARTING_FEN,
    "Trung cuộc cân bằng (Italian)": "r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4",
    "Tàn cuộc Tốt thông e6 + Cặp Tượng": "4k3/p7/p3P3/8/8/8/8/2B1K1B1 w - - 0 1",
}

eval_rows = []
for label, fen in sample_fens.items():
    b = chess.Board(fen)
    eval_rows.append({
        "Thế cờ": label,
        "Evaluate (cp)": round(evaluate_board(b), 2),
        "Pawn Struct Δ": pawn_structure_score(b, chess.WHITE) - pawn_structure_score(b, chess.BLACK),
        "Rook File Δ": rook_file_score(b, chess.WHITE) - rook_file_score(b, chess.BLACK),
        "Bishop Pair Δ": bishop_pair_score(b, chess.WHITE) - bishop_pair_score(b, chess.BLACK),
    })

display(pd.DataFrame(eval_rows))

,Thế cờ,Evaluate (cp),Pawn Struct Δ,Rook File Δ,Bishop Pair Δ
0,Khai cuộc chuẩn,0.00,0,0,0
1,Trung cuộc cân bằng (Italian),31.00,0,0,0
2,Tàn cuộc Tốt thông e6 + Cặp Tượng,722.55,70,0,35


## 2. So Sánh Thời Gian & Số Nút Duyệt Giữa Level 1 – Level 4

In [3]:
probe_fen = "r1bqkbnr/pppp1ppp/2n5/4p3/2B1P3/5N2/PPPP1PPP/RNBQK2R w KQkq - 4 4"
bench_rows = []
for aid in ("level1", "level2", "level3", "level4"):
    env = ChessEnv(fen=probe_fen)
    ag = create_agent(aid)
    m = ag.select_move(env, max_workload=400)
    st = ag.last_search_stats
    t_sec = max(1e-6, float(st.get("time_sec", 0.0)))
    nodes = float(st.get("nodes", 0.0))
    sims = float(st.get("simulations", 0.0))
    bench_rows.append({
        "Agent": ag.name,
        "Nước chọn": env.board.san(m),
        "Nodes": int(nodes),
        "Simulations": int(sims),
        "Time (ms)": round(t_sec * 1000.0, 2),
        "Throughput (NPS/SPS)": round((sims if sims > 0 else nodes) / t_sec, 1),
    })

display(pd.DataFrame(bench_rows))

,Agent,Nước chọn,Nodes,Simulations,Time (ms),Throughput (NPS/SPS)
0,Level1_Random,Bb5,1,0,0.11,8779.6
1,Level2_Heuristic,d4,33,0,17.79,1854.5
2,Level3_AlphaBeta,Nc3,400,0,23.72,16865.5
3,Level4_MCTS_UCT,Nd4,400,400,1277.49,313.1


## 3. Thí Nghiệm Đối Chứng Level 4: `rollout_mode="see_tactical"` vs `rollout_mode="uniform"`

Chạy đối đầu ghép cặp đối xứng trên bộ khai cuộc cân bằng `resources/openings_50.epd` giữa:
- **L4 (`see_tactical`):** Rollout ưu tiên 70% nước bắt quân không lỗ (`see_capture >= 0`) hoặc phong Hậu.
- **L4 (`uniform`):** Rollout ngẫu nhiên đồng đều hoàn toàn.

Đo tỷ lệ điểm $S$, $W/D/L$, khoảng tin cậy Wilson/Bootstrap 95%, độ trễ trung bình và tốc độ mô phỏng (`SPS`).

In [4]:
NUM_OPENINGS = 25  # 25 khai cuộc x 2 màu = 50 ván ghép cặp chuẩn hóa
N_SIM = 100
SEEDS = [42]

openings = load_epd_file(PROJECT_ROOT / "resources" / "openings_50.epd")[:NUM_OPENINGS]

factory_see = partial(Level4MCTSAgent, simulations=N_SIM, rollout_mode="see_tactical", name="Level4_UCT_SEE_Tactical")
factory_uniform = partial(Level4MCTSAgent, simulations=N_SIM, rollout_mode="uniform", name="Level4_UCT_Uniform")

res_rollout = run_paired_matchup(
    agent_a=factory_see(),
    agent_b=factory_uniform(),
    openings=openings,
    seeds=SEEDS,
    budget_mode="workload_control",
    workload_a=N_SIM,
    workload_b=N_SIM,
    max_plies=160,
    bootstrap_resamples=5000,
    workers=4,
    factory_a=factory_see,
    factory_b=factory_uniform,
)

# Lưu kết quả để tái sử dụng cho báo cáo
out_json = PROJECT_ROOT / "results" / "ablation2" / "l4_rollout_ablation.json"
out_json.parent.mkdir(parents=True, exist_ok=True)
out_json.write_text(json.dumps(res_rollout, ensure_ascii=False, indent=2), encoding="utf-8")

display(pd.DataFrame([{
    "Cặp đấu": f"{res_rollout['agent_a']} vs {res_rollout['agent_b']}",
    "Số ván": res_rollout["games"],
    "W/D/L": f"{res_rollout['wins']}/{res_rollout['draws']}/{res_rollout['losses']}",
    "Điểm S (%)": round(res_rollout["score"] * 100.0, 1),
    "Wilson 95% CI": f"[{res_rollout['wilson_ci_95'][0]*100:.1f}%, {res_rollout['wilson_ci_95'][1]*100:.1f}%]",
    "L4 SEE Avg ms": round(res_rollout["agent_a_avg_latency_ms"], 1),
    "L4 Uniform Avg ms": round(res_rollout["agent_b_avg_latency_ms"], 1),
}]))


,Cặp đấu,Số ván,W/D/L,Điểm S (%),Wilson 95% CI,L4 SEE Avg ms,L4 Uniform Avg ms
0,Level4_UCT_SEE_Tactical vs Level4_UCT_Uniform,50,19/19/12,57.0,"[43.3%, 69.7%]",350.9,341.2
